 # AI Research Agent
# Environment & API Key 
# python libraries import
# Gemini model
# Tavily web search tool
# Weather Tool
# AI Agent
# Test Agent

In [105]:
# Import Libraries
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults

# Environment & API Configuration

In [106]:
# Load environment
load_dotenv(".env",override=True)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

# Test Gemini API

In [107]:
import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)
model=genai.GenerativeModel("gemini-3.5-flash")
response=model.generate_content("What is the capital of pakistan")
print(response.text)

The capital of Pakistan is **Islamabad**.


# Tavily Web Search Tool

In [108]:
search_tool = TavilySearchResults(max_results=5,
tavily_api_key=TAVILY_API_KEY)

In [109]:
# Test Web Search

result = search_tool.invoke("What is the capital of France ?")
result

[{'url': 'https://www.britannica.com/place/France',
  'content': "Historically, the Francien dialect became the official language in 1539, eventually replacing Latin and other dialects. Although regional languages were once discouraged, they have been reintroduced in some schools because several have maintained literary traditions. Today, French is considered one of the most internationally significant Romance languages. \n\n 3 Britannica Sources \n    1.   France: Country Facts\n    2.   Valentin Conrart\n    3.   French language: History\n\n This answer is created from Britannica articles using AI. AI can make mistakes, so verify using Britannica articles. \n\n   What is the capital city of France and why is it famous?  \n\nThe capital of France is Paris. Situated in the north-central part of the country, Paris is France's center of commerce and culture. [...] The capital and by far the most important city of France is Paris, one of the world’s preeminent cultural and commercial cent

# Gemini LLM

In [110]:
# Gemini LLM
llm_gemini=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    max_retries=0
)

In [111]:
# prompt
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
You are a helpful AI assistant.

Use weather_tool for weather-related questions.
Use tavily_search_results_json for general web research and latest information.

Available tools:
{tools}

Tools names:
{tool_names}

Use only one tool when needed.

Question: {input}

Thought: {agent_scratchpad}
""")

# Agent Tools

In [112]:
tools = [search_tool,weather_tool]

# AI Agent

In [113]:
agent_executor = agents.initialize_agent(tools,llm,
agent=agents.AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [114]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    max_retries=0,
convert_system_message_to_human=True    
)

# Test AI Agent

In [115]:
response=agent_executor.invoke({
    "input": "What is the capital of india?"
})
print(response["output"])



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "Final Answer",
  "action_input": "The capital of India is New Delhi."
}
```

> Finished chain.
The capital of India is New Delhi.


# Agent Query Function

In [116]:
def ask_agent(question):
    response= agent_executor.invoke({
        "input": question
    })
    return response["output"]

# Agent Test

In [117]:
answer = ask_agent("what is the capital of india?")
print(answer)



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "Final Answer",
  "action_input": "The capital of India is New Delhi."
}
```

> Finished chain.
The capital of India is New Delhi.


# Weather Tool

In [118]:
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """
    import requests

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    current = data["current"]

    return (
        f"Weather in {city}: "
        f"{current['temperature']} C, "
        f"{current['weather_descriptions'][0]}, "
        f"Humidity: {current['humidity']}%, "
        f"Wind Speed: {current['wind_speed']} km/h"
    )

# Test Weather Tool

In [119]:
print(get_weather_data("delhi"))

Weather in delhi: 17 C, Sunny, Humidity: 86%, Wind Speed: 7 km/h


In [120]:
from langchain_core.tools import tool
weather_tool = tool(get_weather_data)

# Test Langchain weather tool

In [121]:
result = weather_tool.invoke({"city": "Delhi"})
print(result)

Weather in Delhi: 17 C, Sunny, Humidity: 86%, Wind Speed: 7 km/h


# Test Agent with weather Query

In [122]:
response = agent_executor.invoke({
    "input" : "What is the current weather in Bangalore?"
})
print(response["output"])



> Entering new AgentExecutor chain...
Action:
```json
{
  "action": "get_weather_data",
  "action_input": {
    "city": "Bangalore"
  }
}
```

Observation: Weather in Bangalore: 30 C, Clear , Humidity: 35%, Wind Speed: 4 km/h
Thought:Action:
```json
{
  "action": "Final Answer",
  "action_input": "The current weather in Bangalore is 30°C and Clear, with a humidity of 35% and a wind speed of 4 km/h."
}
```

> Finished chain.
The current weather in Bangalore is 30°C and Clear, with a humidity of 35% and a wind speed of 4 km/h.
